# Week 28, Day 3: __init__ and attributes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w028d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Kettle:
    def __init__(self, capacity, full=False):
        print("making a kettle")
        self.capacity = capacity
        self.full = full

k1 = Kettle(1.7)
k2 = Kettle(capacity=1.0, full=True)
print(k1.capacity, k1.full, k2.full)

**Predict before you run:** what will this print?



In [ ]:
class Thing:
    def __init__(self, size):
        size = size * 2          # changes only the local variable
        self.size = size

t = Thing(5)
print(t.size)
try:
    print(size)
except NameError:
    print("size was local to __init__")

In [ ]:
class Alarm:
    def __init__(self, hour, minute=0, *, repeat=False, sound="beep"):
        self.hour = hour
        self.minute = minute
        self.repeat = repeat
        self.sound = sound

a = Alarm(7, 30, repeat=True)
b = Alarm(6)
print(a.hour, a.minute, a.repeat, a.sound)
print(b.hour, b.minute, b.repeat)
try:
    Alarm(7, 30, True)
except TypeError as e:
    print("TypeError:", e)

In [ ]:
class Temperature:
    def __init__(self, celsius, place):
        if celsius < -273.15:
            raise ValueError(f"below absolute zero: {celsius}")
        place = " ".join(place.split()).title()
        if not place:
            raise ValueError("place is missing")
        self.celsius = celsius
        self.place = place

t = Temperature(21.5, "  ibadan  ")
print(t.place, t.celsius)
for bad in [(-300, "Leeds"), (10, "   ")]:
    try:
        Temperature(*bad)
    except ValueError as e:
        print("ValueError:", e)

**Predict before you run:** what will this print?



In [ ]:
class Team:
    def __init__(self, name, players=[]):
        self.name = name
        self.players = players

a = Team("Ants")
b = Team("Bees")
a.players.append("Ada")
print(b.players, a.players is b.players)

**Predict before you run:** what will this print?



In [ ]:
class Team:
    def __init__(self, players):
        self.players = players

names = ["Ada", "Bo"]
t = Team(names)
names.append("Cy")
print(t.players)

**Predict before you run:** what will this print?



In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.area = width * height

r = Rectangle(2, 3)
r.width = 10
print(r.area)

> **Key idea.** `__init__` gets the new instance as `self` and sets up its attributes; it returns nothing. Use defaults and keyword-only parameters (`*`) for options. Check and tidy arguments before assigning, raise `ValueError` for bad values, and set every attribute there. Never use a mutable default; copy lists you're given; and don't store values that depend on other attributes.

### Exercise: A checked rectangle

Write `Rectangle(width, height)` storing both, and raising `ValueError` if either isn't more than 0. Add methods `area()` and `perimeter()`, and `is_square()`.

In [ ]:
class Rectangle:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    R = ns[\"Rectangle\"]\n    r = R(3, 4)\n    assert (r.width, r.height, r.area(), r.perimeter(), r.is_square()) == (3, 4, 12, 14, False)\n    assert R(2.5, 2.5).is_square() and R(2.5, 2.5).area() == 6.25\n    r.width = 10\n    assert r.area() == 40, \"work the area out when asked, so it's never stale\"\ndef test_errors():\n    \"bad sizes\"\n    for w, h in [(0, 1), (1, -2), (-1, -1)]:\n        try:\n            ns[\"Rectangle\"](w, h)\n            assert False, f\"Rectangle({w}, {h}) should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Tidy on the way in

Write `Contact(name, email, phone=None)`. `__init__` tidies and checks:

- `name`: runs of spaces collapsed, stripped, each word capitalised (`" ada   OBI "` becomes `"Ada Obi"`); empty raises `ValueError`;
- `email`: stripped and lower-cased; it must contain exactly one `@` with text on both sides, or `ValueError`;
- `phone`: `None`, or kept as its digits only (`"0803 123-4567"` becomes `"08031234567"`).

In [ ]:
class Contact:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tidy():\n    \"tidied\"\n    c = ns[\"Contact\"](\" ada   OBI \", \" Ada@Example.COM \", \"0803 123-4567\")\n    assert (c.name, c.email, c.phone) == (\"Ada Obi\", \"ada@example.com\", \"08031234567\")\n    assert ns[\"Contact\"](\"bo\", \"b@x.ng\").phone is None\ndef test_errors():\n    \"checked\"\n    for args in [(\"   \", \"a@b.c\"), (\"Ada\", \"no-at-sign\"), (\"Ada\", \"a@@b\"), (\"Ada\", \"@b.c\"), (\"Ada\", \"a@\")]:\n        try:\n            ns[\"Contact\"](*args)\n            assert False, f\"Contact{args} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: A playlist that keeps its own list

Write `Playlist(name, songs=None)`, keeping its **own** list of songs (empty if none given), with `add(song)` and `total()` (the number of songs). Two playlists must never share a list, and changing the list you passed in must not change the playlist.

In [ ]:
class Playlist:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_separate():\n    \"separate lists\"\n    P = ns[\"Playlist\"]\n    a, b = P(\"Road\"), P(\"Rain\")\n    a.add(\"Song A\")\n    assert b.total() == 0 and a.total() == 1\ndef test_copy():\n    \"copies what it's given\"\n    P = ns[\"Playlist\"]\n    mine = [\"x\", \"y\"]\n    p = P(\"Mine\", mine)\n    mine.append(\"z\")\n    p.add(\"w\")\n    assert p.total() == 3 and mine == [\"x\", \"y\", \"z\"]\n    t = (\"q\",)\n    assert P(\"T\", t).total() == 1")

### Exercise: Keyword-only options

Write `Timer(minutes, *, label="Timer", loud=False)`. Store all three. `minutes` must be a whole number from 1 to 180, or `ValueError`. Then make `t = Timer(25, label="Focus")` and print `Focus: 25 min, loud=False`.

In [ ]:
class Timer:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"output\"\n    expect_output(\"Focus: 25 min, loud=False\")\ndef test_rules():\n    \"rules\"\n    T = ns[\"Timer\"]\n    assert T(180).label == \"Timer\" and T(1, loud=True).loud is True\n    for bad in (0, 181, -5):\n        try:\n            T(bad)\n            assert False, f\"Timer({bad}) should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        T(5, \"Tea\")\n        assert False, \"label should be keyword-only\"\n    except TypeError:\n        pass")

### Bug hunt: Shared by accident

Every `Order` is sharing one list of items, so the second order has the first order's items in it. Fix it.

In [ ]:
class Order:
    def __init__(self, customer, items=[]):
        self.customer = customer
        self.items = items

    def add(self, item):
        self.items.append(item)

a = Order("Ada")
a.add("tea")
b = Order("Bo")
print(b.items)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Bo's order is empty\"\n    expect_output(\"[]\")\n    O = ns[\"Order\"]\n    given = [\"x\"]\n    o = O(\"Cy\", given)\n    o.add(\"y\")\n    assert given == [\"x\"], \"copy the list you're given, too\"")

### Bug hunt: Where did it go?

This crashes with `AttributeError: 'Pet' object has no attribute 'name'`. Fix `__init__`.

In [ ]:
class Pet:
    def __init__(self, name, kind):
        name = name.title()
        kind = kind.lower()

    def describe(self):
        return f"{self.name} the {self.kind}"

print(Pet("rex", "DOG").describe())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"described\"\n    expect_output(\"Rex the dog\")")

### Bug hunt: A stale total

After adding an item, `basket.total` is still 0. Replace the stored total with a method `total()` that works it out from the items each time, and fix the last line to call it.

In [ ]:
class Basket:
    def __init__(self):
        self.prices = []
        self.total = sum(self.prices)

    def add(self, price):
        self.prices.append(price)

basket = Basket()
basket.add(250)
basket.add(120)
print(basket.total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the total\"\n    expect_output(\"370\")\n    import inspect\n    assert inspect.ismethod(ns[\"basket\"].total), \"total should be a method now\"")

### Puzzle: Check, then assign

Arrange the lines into a `Score` class whose `__init__` checks the value is 0 to 100 before storing it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Jar:
    def __init__(self, sweets=None):
        self.sweets = sweets if sweets is not None else []

a = Jar()
b = Jar(["mint"])
c = Jar(b.sweets)
c.sweets.append("toffee")

p = ___   # a.sweets
q = ___   # b.sweets
r = ___   # b.sweets is c.sweets
s = ___   # what Jar.__init__ returns (write the Python value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [] and q == [\"mint\", \"toffee\"] and r is True and s is None")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: fractions kept tidy

Write `Frac(top, bottom=1)` for fractions of whole numbers, always stored in **lowest terms** with a **positive** bottom: `Frac(6, -8)` is stored as top -3, bottom 4. A bottom of 0 raises `ZeroDivisionError`; anything that isn't an `int` raises `TypeError`. Add methods `plus(other)` and `times(other)` returning new `Frac`s, `value()` returning the float, and `text()` returning `"-3/4"`, or just `"5"` when the bottom is 1.

In [ ]:
from math import gcd

class Frac:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tidy():\n    \"lowest terms, positive bottom\"\n    F = ns[\"Frac\"]\n    f = F(6, -8)\n    assert (f.top, f.bottom) == (-3, 4) and f.text() == \"-3/4\"\n    assert F(0, 5).text() == \"0\" and (F(0, -5).top, F(0, -5).bottom) == (0, 1) and F(10, 5).text() == \"2\" and F(7).text() == \"7\"\ndef test_maths():\n    \"plus, times, value\"\n    F = ns[\"Frac\"]\n    assert F(1, 2).plus(F(1, 3)).text() == \"5/6\" and F(1, 2).plus(F(1, 2)).text() == \"1\"\n    assert F(2, 3).times(F(9, -4)).text() == \"-3/2\" and F(3, 4).value() == 0.75\n    a = F(1, 2)\n    a.plus(F(1, 2))\n    assert a.text() == \"1/2\", \"plus should return a new Frac, not change this one\"\ndef test_errors():\n    \"errors\"\n    F = ns[\"Frac\"]\n    for args, err in [((1, 0), ZeroDivisionError), ((1.5, 2), TypeError), ((\"1\", 2), TypeError), ((1, True), TypeError)]:\n        try:\n            F(*args)\n            assert False, f\"Frac{args} should raise {err.__name__}\"\n        except err:\n            pass")

## Warm-up

`__init__` runs by itself every time an instance is made: "making a kettle" is printed twice. Its parameters work exactly like any function's, defaults and keyword arguments included.

## How __init__ is called

When you write `Kettle(1.7)`, Python:

1. makes a new, empty `Kettle` object;
2. calls `__init__` with that new object as `self`, and your arguments after it: `Kettle.__init__(new_object, 1.7)`;
3. gives you the object.

So inside `__init__`, `self` is the instance being set up, and `self.capacity = capacity` creates an attribute on it. The parameter `capacity` is just a local variable that disappears when `__init__` ends; the **attribute** `self.capacity` is what lasts.

`__init__` must not return anything (it returns `None`, like any function without `return`). Returning a value raises `TypeError`. The new object is what you get back from calling the class, not anything `__init__` returns.

The double underscores mark a **special method** ("dunder" method, for **d**ouble **under**score): one Python calls for you at a particular moment. You'll meet many more next week.

## Defaults and keywords

Give parameters defaults for values that are usually the same, and call with keywords when the meaning of a bare value isn't obvious:

```
Kettle(1.7, True)                    # what does True mean?
Kettle(capacity=1.7, full=True)      # clear
```

A useful habit for classes with several optional settings is to make them **keyword-only**, by putting a `*` in the parameter list: everything after it must be given by name, so callers can't mix them up.

## Valid from the start

An object that starts in a broken state causes bugs far from where it was made. `__init__` is the place to **check** arguments, raising an error for nonsense, and to **tidy** them, so every instance follows the same rules (the "check at the edge" idea from Week 25, applied to objects).

Check first, then assign: if a check fails, the half-made object is thrown away and never reaches the caller. Raise `ValueError` for a value of the right type that's not allowed, and `TypeError` for the wrong type altogether (if you check types at all; often it's fine to let Python fail naturally).

Set **every** attribute in `__init__`, even ones that start as `None` or empty. Then anyone reading the class sees its whole shape in one place, and no method ever finds an attribute missing.

## Three traps

**1. A mutable default.** A default value is made **once**, when the `def` runs, and shared by every call that uses it. So a list default is the same list for every instance:

The fix is the one from Week 26's review: default to `None`, and make a new list inside.

```
def __init__(self, name, players=None):
    self.name = name
    self.players = [] if players is None else list(players)
```

**2. Keeping the caller's list.** `self.players = players` stores a reference to the list the caller passed in. If the caller later changes their list, your object changes too, and the other way round. Copy what you're given with `list(players)` (or `dict(...)`, `set(...)`), as in the fix above.

**3. An attribute that goes out of date.** It's tempting to work out values in `__init__` and store them:

The stored `area` doesn't know the width changed. Store the **basic facts**, and work out values that depend on them when they're asked for, with a method:

```
def area(self):
    return self.width * self.height
```

Next week, **properties** let `r.area` (without brackets) work out the value each time, so it can never be stale.

## Practice

## Quiz

## Daily challenge

## Before you go

Look back at the classes you wrote yesterday. Does each one check its arguments? Does any keep a caller's list, or store a value that could go stale? Tomorrow: methods and `self` in depth, including methods that call other methods.